# Normalizations Practice

Normalization:

- Purpose: Stabilize training, reduce internal covariate shift
- Examples: BatchNorm, LayerNorm, InstanceNorm, GroupNorm
- When: During both training and inference
- Effect: Normalize activations to have mean≈0, std≈1

| Normalization | Normalize Over | Use Case |
|----------------|----------------|-----------|
| **BatchNorm** | Batch + spatial dims | CNNs with large batches |
| **LayerNorm** | All features per sample | Transformers, RNNs |
| **InstanceNorm** | Spatial dims per channel | Style transfer, GANs |
| **GroupNorm** | Groups of channels | Small batches, alternative to BatchNorm |

Important Concepts:
Training vs Inference:
- Training: Calculate statistics from current batch
- Inference: Use running statistics (accumulated during training)

Broadcasting:
- Reshape gamma/beta to match input dimensions
- Example: For (B, C, H, W), reshape to (1, C, 1, 1)

Running Statistics:
- Only BatchNorm uses them (others don't need them)
- Updated with exponential moving average

In [8]:
import torch
import torch.nn as nn

# BatchNorm1d

https://docs.pytorch.org/docs/stable/generated/torch.nn.BatchNorm1d.html

In [13]:
# ============================================================================
# PRACTICE PROBLEM 1: BatchNorm1d
# ============================================================================

class BatchNorm1d(nn.Module):
    """
    Implements BatchNorm1d (Batch Normalization for 1D data).
    
    Normalizes across the batch dimension for each feature independently.
    Formula: y = (x - mean) / sqrt(var + eps) * gamma + beta
    
    Where mean and var are computed across the batch dimension.
    """
    
    num_features: int
    eps: float
    momentum: float
    
    def __init__(self, num_features: int, eps: float = 1e-5, 
                 momentum: float = 0.1) -> None:
        """
        Args:
            num_features: Number of features (C from input shape).
            eps: Small value for numerical stability.
            momentum: Momentum for running statistics.
        
        Returns:
            None.
        """
        super().__init__()
        self.num_features = num_features
        self.eps = eps
        self.momentum = momentum
        
        # Learnable parameters
        self.gamma = nn.Parameter(torch.ones(num_features))  # Scale
        self.beta = nn.Parameter(torch.zeros(num_features))  # Shift
        
        # Running statistics (not learnable, updated during training)
        self.register_buffer('running_mean', torch.zeros(num_features))
        self.register_buffer('running_var', torch.ones(num_features))
        
        return None
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input tensor. Dimensions: [batch, num_features] or 
               [batch, num_features, length].
        
        Returns:
            Normalized tensor. Same dimensions as input.
        
        Hint:
        1. If training:
           - Calculate mean and var across batch dimension (dim=0)
           - For 3D input (B, C, L): mean/var shape should be (C,)
           - Update running_mean and running_var with momentum
        2. If eval (inference):
           - Use running_mean and running_var
        3. Normalize: (x - mean) / sqrt(var + eps)
        4. Scale and shift: normalized * gamma + beta
        5. Handle broadcasting correctly!
        
        For 2D input (B, C): mean/var over dim 0 → shape (C,)
        For 3D input (B, C, L): mean/var over dims (0, 2) → shape (C,)
        """
        # TODO
        if self.training:
            if x.dim() == 2:
                mean = x.mean(dim=(0))
                var = x.var(dim=0, unbiased=False)
                
            else: 
                mean = x.mean(dim=(0, 2))
                var = x.var(dim=(0, 2), unbiased=False)
                                            
            self.running_mean =(1 - self.momentum) * self.running_mean + self.momentum * mean
            self.running_var =(1 - self.momentum) * self.running_var + self.momentum * var
            
        else:
            mean = self.running_mean
            var = self.running_var
            
        x_normalized =  (x-mean) / torch.sqrt(var+self.eps)
        
        if x.dim() == 2:
            return x_normalized * self.gamma + self.beta
        else:  # 3D
            return x_normalized * self.gamma.view(1, -1, 1) + self.beta.view(1, -1, 1)
        
           


# Test your implementation
def test_batchnorm1d():
    # Test 2D input (batch, features)
    x = torch.randn(4, 3)  # 4 samples, 3 features
    
    my_bn = BatchNorm1d(num_features=3)
    pytorch_bn = nn.BatchNorm1d(num_features=3)
    
    # Copy parameters to ensure same initialization
    pytorch_bn.weight.data = my_bn.gamma.data.clone()
    pytorch_bn.bias.data = my_bn.beta.data.clone()
    
    # Set to training mode
    my_bn.train()
    pytorch_bn.train()
    
    result = my_bn(x)
    expected = pytorch_bn(x)
    
    print(f"Input shape: {x.shape}")
    print(f"Your result:\n{result}")
    print(f"Expected:\n{expected}")
    print(f"Match: {torch.allclose(result, expected, atol=1e-5)}\n")


In [14]:
test_batchnorm1d()

Input shape: torch.Size([4, 3])
Your result:
tensor([[ 1.5756, -0.3448,  0.1858],
        [-0.9728,  0.5337,  1.4680],
        [ 0.1399,  1.2431, -0.3563],
        [-0.7426, -1.4321, -1.2975]], grad_fn=<AddBackward0>)
Expected:
tensor([[ 1.5756, -0.3448,  0.1858],
        [-0.9728,  0.5337,  1.4680],
        [ 0.1399,  1.2431, -0.3563],
        [-0.7426, -1.4321, -1.2975]], grad_fn=<NativeBatchNormBackward0>)
Match: True



# BatchNorm2d

In [ ]:
# ============================================================================
# PRACTICE PROBLEM 2: BatchNorm2d
# ============================================================================

class BatchNorm2d(nn.Module):
    """
    Implements BatchNorm2d (Batch Normalization for 2D data - images).
    
    Normalizes across batch and spatial dimensions (H, W) for each channel.
    Input shape: (B, C, H, W)
    Statistics computed over: (B, H, W) → one mean/var per channel
    """
    
    num_features: int
    eps: float
    momentum: float
    
    def __init__(self, num_features: int, eps: float = 1e-5,
                 momentum: float = 0.1) -> None:
        """
        Args:
            num_features: Number of channels (C from input shape).
            eps: Small value for numerical stability.
            momentum: Momentum for running statistics.
        
        Returns:
            None.
        """
        super().__init__()
        self.num_features = num_features
        self.eps = eps
        self.momentum = momentum
        
        # Learnable parameters (one per channel)
        self.gamma = nn.Parameter(torch.ones(num_features))
        self.beta = nn.Parameter(torch.zeros(num_features))
        
        # Running statistics
        self.register_buffer('running_mean', torch.zeros(num_features))
        self.register_buffer('running_var', torch.ones(num_features))
        
        return None
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input tensor. Dimensions: [batch, channels, height, width].
        
        Returns:
            Normalized tensor. Same dimensions as input.
        
        Hint:
        1. Input shape: (B, C, H, W)
        2. If training:
           - Calculate mean/var over dims (0, 2, 3) → shape (C,)
           - Update running statistics
        3. If eval: use running statistics
        4. Normalize: (x - mean) / sqrt(var + eps)
        5. Reshape gamma and beta to (1, C, 1, 1) for broadcasting
        6. Scale and shift: normalized * gamma + beta
        """
        # TODO
        pass


# Test your implementation
def test_batchnorm2d():
    x = torch.randn(2, 3, 4, 4)  # 2 images, 3 channels, 4×4
    
    my_bn = BatchNorm2d(num_features=3)
    pytorch_bn = nn.BatchNorm2d(num_features=3)
    
    # Copy parameters
    pytorch_bn.weight.data = my_bn.gamma.data.clone()
    pytorch_bn.bias.data = my_bn.beta.data.clone()
    
    my_bn.train()
    pytorch_bn.train()
    
    result = my_bn(x)
    expected = pytorch_bn(x)
    
    print(f"Input shape: {x.shape}")
    print(f"Your result shape: {result.shape}")
    print(f"Match: {torch.allclose(result, expected, atol=1e-5)}\n")




In [ ]:
test_batchnorm2d()

# LayerNorm

In [19]:
# ============================================================================
# PRACTICE PROBLEM 3: LayerNorm
# ============================================================================

class LayerNorm(nn.Module):
    """
    Implements Layer Normalization.
    
    Unlike BatchNorm (normalizes across batch), LayerNorm normalizes 
    across features for each sample independently.
    
    Common in Transformers and RNNs.
    """
    
    normalized_shape: tuple
    eps: float
    
    def __init__(self, normalized_shape: int or tuple, eps: float = 1e-5) -> None:
        """
        Args:
            normalized_shape: Shape of the input features to normalize.
                             Can be int or tuple.
            eps: Small value for numerical stability.
        
        Returns:
            None.
        """
        super().__init__()
        if isinstance(normalized_shape, int):
            normalized_shape = (normalized_shape,)
        self.normalized_shape = normalized_shape
        self.eps = eps
        
        # Learnable parameters (same shape as normalized_shape)
        self.gamma = nn.Parameter(torch.ones(normalized_shape))
        self.beta = nn.Parameter(torch.zeros(normalized_shape))
        
        return None
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input tensor. Last dimension(s) must match normalized_shape.
        
        Returns:
            Normalized tensor. Same dimensions as input.
        
        Hint:
        1. Calculate dimensions to normalize over (last len(normalized_shape) dims)
        2. Calculate mean and var over those dimensions
        3. Use keepdim=True to maintain dimensions for broadcasting
        4. Normalize: (x - mean) / sqrt(var + eps)
        5. Scale and shift: normalized * gamma + beta
        
        Example: 
        - Input shape: (B, L, D), normalized_shape=(D,)
        - Normalize over dim=-1 (last dimension)
        - Mean/var shape: (B, L, 1)
        """
        # TODO
        mean = torch.mean(x, dim=(1,2), keepdim=True)
        var = torch.var(x, dim=(1,2), keepdim=True, unbiased=False)
        
        x_normalized = (x-mean) / torch.sqrt(var+self.eps)
        
        return x_normalized * self.gamma + self.beta


# Test your implementation
def test_layernorm():
    x = torch.randn(2, 5, 10)  # (batch, sequence, features)
    
    my_ln = LayerNorm(normalized_shape=10)
    pytorch_ln = nn.LayerNorm(normalized_shape=10)
    
    # Copy parameters
    pytorch_ln.weight.data = my_ln.gamma.data.clone()
    pytorch_ln.bias.data = my_ln.beta.data.clone()
    
    result = my_ln(x)
    expected = pytorch_ln(x)
    
    print(f"Input shape: {x.shape}")
    print(f"Your result shape: {result.shape}")
    print(f"Match: {torch.allclose(result, expected, atol=1e-5)}\n")

In [21]:
test_layernorm()

Input shape: torch.Size([2, 5, 10])
Your result shape: torch.Size([2, 5, 10])
Match: False



In [22]:
# ============================================================================
# PRACTICE PROBLEM 4: InstanceNorm2d
# ============================================================================

class InstanceNorm2d(nn.Module):
    """
    Implements Instance Normalization for 2D data.
    
    Normalizes each channel of each sample independently (per instance).
    Input shape: (B, C, H, W)
    Statistics computed over: (H, W) for each (B, C) independently
    
    Used in style transfer and GANs.
    """
    
    num_features: int
    eps: float
    
    def __init__(self, num_features: int, eps: float = 1e-5) -> None:
        """
        Args:
            num_features: Number of channels (C from input shape).
            eps: Small value for numerical stability.
        
        Returns:
            None.
        """
        super().__init__()
        self.num_features = num_features
        self.eps = eps
        
        # Learnable parameters (one per channel)
        self.gamma = nn.Parameter(torch.ones(num_features))
        self.beta = nn.Parameter(torch.zeros(num_features))
        
        return None
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input tensor. Dimensions: [batch, channels, height, width].
        
        Returns:
            Normalized tensor. Same dimensions as input.
        
        Hint:
        1. Input shape: (B, C, H, W)
        2. Calculate mean/var over dims (2, 3) → shape (B, C)
        3. Use keepdim=True → mean/var shape becomes (B, C, 1, 1)
        4. Normalize: (x - mean) / sqrt(var + eps)
        5. Reshape gamma/beta to (1, C, 1, 1) for broadcasting
        6. Scale and shift: normalized * gamma + beta
        
        Key difference from BatchNorm:
        - BatchNorm: mean/var over (B, H, W) → one per channel
        - InstanceNorm: mean/var over (H, W) → one per (batch, channel)
        """
        # TODO
        mean = torch.mean(x, dim=(2,3), keepdim=True)
        var = torch.var(x, dim=(2,3), keepdim=True, unbiased=False)
        
        normalized_x = (x-mean) / torch.sqrt(var + self.eps)
        
        return normalized_x * self.gamma +self.beta
        
        


# Test your implementation
def test_instancenorm2d():
    x = torch.randn(2, 3, 4, 4)  # 2 images, 3 channels, 4×4
    
    my_in = InstanceNorm2d(num_features=3)
    pytorch_in = nn.InstanceNorm2d(num_features=3, affine=True)
    
    # Copy parameters
    pytorch_in.weight.data = my_in.gamma.data.clone()
    pytorch_in.bias.data = my_in.beta.data.clone()
    
    result = my_in(x)
    expected = pytorch_in(x)
    
    print(f"Input shape: {x.shape}")
    print(f"Your result shape: {result.shape}")
    print(f"Match: {torch.allclose(result, expected, atol=1e-5)}\n")

# test_instancenorm2d()


# ============================================================================
# PRACTICE PROBLEM 5: GroupNorm (CHALLENGING!)
# ============================================================================

class GroupNorm(nn.Module):
    """
    Implements Group Normalization.
    
    Divides channels into groups and normalizes within each group.
    Middle ground between LayerNorm (1 group) and InstanceNorm (C groups).
    
    Input shape: (B, C, H, W)
    Groups channels: C → G groups of C/G channels each
    Normalizes over (C/G, H, W) for each (B, G)
    """
    
    num_groups: int
    num_channels: int
    eps: float
    
    def __init__(self, num_groups: int, num_channels: int, 
                 eps: float = 1e-5) -> None:
        """
        Args:
            num_groups: Number of groups to divide channels into.
            num_channels: Number of channels (C from input shape).
            eps: Small value for numerical stability.
        
        Returns:
            None.
        """
        super().__init__()
        assert num_channels % num_groups == 0, \
            "num_channels must be divisible by num_groups"
        
        self.num_groups = num_groups
        self.num_channels = num_channels
        self.eps = eps
        
        # Learnable parameters (one per channel)
        self.gamma = nn.Parameter(torch.ones(num_channels))
        self.beta = nn.Parameter(torch.zeros(num_channels))
        
        return None
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input tensor. Dimensions: [batch, channels, height, width].
        
        Returns:
            Normalized tensor. Same dimensions as input.
        
        Hint:
        1. Input shape: (B, C, H, W)
        2. Reshape to: (B, G, C/G, H, W) where G = num_groups
        3. Calculate mean/var over dims (2, 3, 4) → shape (B, G)
        4. Reshape mean/var to (B, G, 1, 1, 1) for broadcasting
        5. Normalize the reshaped tensor
        6. Reshape back to (B, C, H, W)
        7. Apply gamma and beta (reshaped to (1, C, 1, 1))
        """
        # TODO
        pass


# Test your implementation
def test_groupnorm():
    x = torch.randn(2, 6, 4, 4)  # 2 images, 6 channels, 4×4
    
    my_gn = GroupNorm(num_groups=3, num_channels=6)  # 3 groups of 2 channels
    pytorch_gn = nn.GroupNorm(num_groups=3, num_channels=6)
    
    # Copy parameters
    pytorch_gn.weight.data = my_gn.gamma.data.clone()
    pytorch_gn.bias.data = my_gn.beta.data.clone()
    
    result = my_gn(x)
    expected = pytorch_gn(x)
    
    print(f"Input shape: {x.shape}")
    print(f"Num groups: {my_gn.num_groups}")
    print(f"Your result shape: {result.shape}")
    print(f"Match: {torch.allclose(result, expected, atol=1e-5)}\n")

# test_groupnorm()


# ============================================================================
# BONUS: Comparison of Normalization Techniques
# ============================================================================

def compare_normalizations():
    """
    Visual comparison of different normalization techniques.
    """
    print("="*70)
    print("NORMALIZATION TECHNIQUES COMPARISON")
    print("="*70)
    
    print("""
    Input shape: (B, C, H, W)
    
    1. BatchNorm2d:
       - Normalize over: (B, H, W) dimensions
       - Statistics per: channel
       - Output: One mean/var per channel across entire batch
       - Use case: Large batch sizes, stable batch statistics
    
    2. LayerNorm:
       - Normalize over: (C, H, W) dimensions  
       - Statistics per: sample
       - Output: One mean/var per sample
       - Use case: Transformers, RNNs, small batches
    
    3. InstanceNorm2d:
       - Normalize over: (H, W) dimensions
       - Statistics per: (batch, channel)
       - Output: One mean/var per channel per sample
       - Use case: Style transfer, GANs
    
    4. GroupNorm:
       - Normalize over: (C/G, H, W) dimensions (within each group)
       - Statistics per: (batch, group)
       - Output: One mean/var per group per sample
       - Use case: Small batches, alternative to BatchNorm
    
    Visual representation for input (2, 4, 3, 3):
    
    BatchNorm2d (4 channels):
        Channel 0: mean/var over all (B=2, H=3, W=3) → 1 mean, 1 var
        Channel 1: mean/var over all (B=2, H=3, W=3) → 1 mean, 1 var
        ...
        → 4 means, 4 vars total
    
    InstanceNorm2d (4 channels):
        Sample 0, Channel 0: mean/var over (H=3, W=3) → 1 mean, 1 var
        Sample 0, Channel 1: mean/var over (H=3, W=3) → 1 mean, 1 var
        Sample 1, Channel 0: mean/var over (H=3, W=3) → 1 mean, 1 var
        ...
        → 2×4 = 8 means, 8 vars total
    
    LayerNorm:
        Sample 0: mean/var over all (C=4, H=3, W=3) → 1 mean, 1 var
        Sample 1: mean/var over all (C=4, H=3, W=3) → 1 mean, 1 var
        → 2 means, 2 vars total
    
    GroupNorm (2 groups):
        Sample 0, Group 0 (channels 0-1): mean/var over (C/G=2, H=3, W=3)
        Sample 0, Group 1 (channels 2-3): mean/var over (C/G=2, H=3, W=3)
        Sample 1, Group 0: ...
        Sample 1, Group 1: ...
        → 2×2 = 4 means, 4 vars total
    """)

# compare_normalizations()


# ============================================================================
# BONUS: Run all tests
# ============================================================================

def run_all_tests():
    """Run all test functions"""
    print("="*70)
    print("RUNNING ALL NORMALIZATION TESTS")
    print("="*70 + "\n")
    
    tests = [
        ("BatchNorm1d", test_batchnorm1d),
        ("BatchNorm2d", test_batchnorm2d),
        ("LayerNorm", test_layernorm),
        ("InstanceNorm2d", test_instancenorm2d),
        ("GroupNorm", test_groupnorm),
    ]
    
    for name, test_func in tests:
        print(f"--- {name} ---")
        try:
            test_func()
        except Exception as e:
            print(f"❌ Error: {e}\n")
    
    print("="*70)
    print("TESTS COMPLETE")
    print("="*70)

# Uncomment to run all tests:
# run_all_tests()
# compare_normalizations()


# ============================================================================
# SOLUTIONS (Don't look until you've tried!)
# ============================================================================

# Scroll down for solutions...
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#

"""
SOLUTIONS:

# 1. BatchNorm1d
def forward(self, x):
    if self.training:
        # Calculate mean and var
        if x.dim() == 2:  # (B, C)
            mean = x.mean(dim=0)
            var = x.var(dim=0, unbiased=False)
        else:  # (B, C, L)
            mean = x.mean(dim=(0, 2))
            var = x.var(dim=(0, 2), unbiased=False)
        
        # Update running statistics
        self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * mean
        self.running_var = (1 - self.momentum) * self.running_var + self.momentum * var
    else:
        mean = self.running_mean
        var = self.running_var
    
    # Normalize
    x_normalized = (x - mean) / torch.sqrt(var + self.eps)
    
    # Scale and shift
    if x.dim() == 2:
        return x_normalized * self.gamma + self.beta
    else:  # 3D
        return x_normalized * self.gamma.view(1, -1, 1) + self.beta.view(1, -1, 1)


# 2. BatchNorm2d
def forward(self, x):
    if self.training:
        # Mean and var over (B, H, W)
        mean = x.mean(dim=(0, 2, 3))
        var = x.var(dim=(0, 2, 3), unbiased=False)
        
        # Update running statistics
        self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * mean
        self.running_var = (1 - self.momentum) * self.running_var + self.momentum * var
    else:
        mean = self.running_mean
        var = self.running_var
    
    # Reshape for broadcasting: (1, C, 1, 1)
    mean = mean.view(1, -1, 1, 1)
    var = var.view(1, -1, 1, 1)
    gamma = self.gamma.view(1, -1, 1, 1)
    beta = self.beta.view(1, -1, 1, 1)
    
    # Normalize, scale, and shift
    x_normalized = (x - mean) / torch.sqrt(var + self.eps)
    return x_normalized * gamma + beta


# 3. LayerNorm
def forward(self, x):
    # Dimensions to normalize over (last len(normalized_shape) dims)
    dims = list(range(-len(self.normalized_shape), 0))
    
    # Calculate mean and var
    mean = x.mean(dim=dims, keepdim=True)
    var = x.var(dim=dims, keepdim=True, unbiased=False)
    
    # Normalize
    x_normalized = (x - mean) / torch.sqrt(var + self.eps)
    
    # Scale and shift
    return x_normalized * self.gamma + self.beta


# 4. InstanceNorm2d
def forward(self, x):
    # Mean and var over (H, W) for each (B, C)
    mean = x.mean(dim=(2, 3), keepdim=True)
    var = x.var(dim=(2, 3), keepdim=True, unbiased=False)
    
    # Normalize
    x_normalized = (x - mean) / torch.sqrt(var + self.eps)
    
    # Scale and shift
    gamma = self.gamma.view(1, -1, 1, 1)
    beta = self.beta.view(1, -1, 1, 1)
    return x_normalized * gamma + beta


# 5. GroupNorm
def forward(self, x):
    B, C, H, W = x.shape
    G = self.num_groups
    
    # Reshape: (B, G, C/G, H, W)
    x_grouped = x.view(B, G, C // G, H, W)
    
    # Mean and var over (C/G, H, W) for each (B, G)
    mean = x_grouped.mean(dim=(2, 3, 4), keepdim=True)
    var = x_grouped.var(dim=(2, 3, 4), keepdim=True, unbiased=False)
    
    # Normalize
    x_normalized = (x_grouped - mean) / torch.sqrt(var + self.eps)
    
    # Reshape back to (B, C, H, W)
    x_normalized = x_normalized.view(B, C, H, W)
    
    # Scale and shift
    gamma = self.gamma.view(1, -1, 1, 1)
    beta = self.beta.view(1, -1, 1, 1)
    return x_normalized * gamma + beta
"""

'\nSOLUTIONS:\n\n# 1. BatchNorm1d\ndef forward(self, x):\n    if self.training:\n        # Calculate mean and var\n        if x.dim() == 2:  # (B, C)\n            mean = x.mean(dim=0)\n            var = x.var(dim=0, unbiased=False)\n        else:  # (B, C, L)\n            mean = x.mean(dim=(0, 2))\n            var = x.var(dim=(0, 2), unbiased=False)\n        \n        # Update running statistics\n        self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * mean\n        self.running_var = (1 - self.momentum) * self.running_var + self.momentum * var\n    else:\n        mean = self.running_mean\n        var = self.running_var\n    \n    # Normalize\n    x_normalized = (x - mean) / torch.sqrt(var + self.eps)\n    \n    # Scale and shift\n    if x.dim() == 2:\n        return x_normalized * self.gamma + self.beta\n    else:  # 3D\n        return x_normalized * self.gamma.view(1, -1, 1) + self.beta.view(1, -1, 1)\n\n\n# 2. BatchNorm2d\ndef forward(self, x):\n    if s